In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from src.db import get_engine
from sqlalchemy import text

engine = get_engine()

ModuleNotFoundError: No module named 'src'

---------------------------------------------------------------------------------------

#Bankda 5000 müştəridən churn olanları tapaq
#Ümumi churn faizi

In [ ]:
sql = """
SELECT
    churned,
    COUNT(*) AS musteri_sayi,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS churn_faizi
FROM bronze.customers
GROUP BY churned
ORDER BY churned
"""

pd.read_sql(sql, engine)

,churned,musteri_sayi,churn_faizi
0,0,3992,79.84
1,1,1008,20.16


In [ ]:
gold_sql = sql.replace("FROM ", "INTO gold.churn_summary FROM ", 1)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

#Aktivlik üzrə churn

In [ ]:
sql = """
SELECT
    CASE
        WHEN is_active = 1 THEN N'Aktivdir'
        ELSE N'Aktiv deyil'
    END AS Aktivlik,
    COUNT(*) AS musteri_sayi,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS geden_musteri,
    ROUND(
        100.0 * SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_faizi
INTO gold.churn_by_activity
FROM bronze.customers
GROUP BY is_active
"""

with engine.begin() as conn:
    conn.execute(text(sql))

In [ ]:
pd.read_sql("SELECT * FROM gold.churn_by_activity", engine)

,Aktivlik,musteri_sayi,geden_musteri,churn_faizi
0,Aktiv deyil,2029,584,28.78
1,Aktivdir,2971,424,14.27


In [ ]:
gold_sql = sql.replace("FROM ", "INTO gold.churn_by_activity FROM ", 1)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

OperationalError: (pymssql.exceptions.OperationalError) (156, b"Incorrect syntax near the keyword 'INTO'.DB-Lib error message 20018, severity 15:\nGeneral SQL Server error: Check messages from the SQL Server\n")
[SQL: 
SELECT
    CASE
        WHEN is_active = 1 THEN N'Aktivdir'
        ELSE N'Aktiv deyil'
    END AS Aktivlik,
    COUNT(*) AS musteri_sayi,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) AS geden_musteri,
    ROUND(
        100.0 * SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_faizi
INTO gold.churn_by_activity
INTO gold.churn_by_activity FROM bronze.customers
GROUP BY is_active
]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

#MƏHSUL SAYINA GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
SELECT
    products_count,
    COUNT(DISTINCT customer_id) AS musteri_sayi,
    COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END) AS geden_musteri,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END)
        / COUNT(DISTINCT customer_id), 2
    ) AS churn_faizi
FROM silver.transactions
GROUP BY products_count
ORDER BY products_count
"""

pd.read_sql(sql, engine)

,products_count,musteri_sayi,geden_musteri,churn_faizi
0,1,2288,519,22.68
1,2,2113,214,10.13
2,3,440,170,38.64
3,4,156,103,66.03


In [ ]:
gold_sql = sql.replace("FROM ", "INTO gold.churn_by_products FROM ", 1)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

# 4. MƏHSUL SAYI VƏ AKTİVLİYƏ GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
SELECT
    products_count,
    is_active,
    COUNT(DISTINCT customer_id) AS musteri_sayi,
    COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END) AS geden_musteri,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END)
        / COUNT(DISTINCT customer_id), 2
    ) AS churn_faizi
FROM silver.transactions
GROUP BY products_count, is_active
"""

pd.read_sql(sql + " ORDER BY products_count, is_active", engine)

,products_count,is_active,musteri_sayi,geden_musteri,churn_faizi
0,1,0,1036,327,31.56
1,1,1,1252,192,15.34
2,2,0,786,133,16.92
3,2,1,1327,81,6.10
4,3,0,145,74,51.03
5,3,1,295,96,32.54
6,4,0,59,48,81.36
7,4,1,97,55,56.70


In [ ]:
gold_sql = sql.replace(
    "FROM silver.transactions",
    "INTO gold.churn_products_activity FROM silver.transactions"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

# 5. ƏMƏLİYYAT SAYINA GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
WITH customer_tx AS (
    SELECT
        customer_id,
        churned,
        COUNT(*) AS emeliyyat_sayi
    FROM silver.transactions
    GROUP BY customer_id, churned
)
SELECT
    churned,
    COUNT(*) AS musteri_sayi,
    ROUND(AVG(emeliyyat_sayi * 1.0), 2) AS orta_emeliyyat_sayi
FROM customer_tx
GROUP BY churned
"""

pd.read_sql(sql, engine)

,churned,musteri_sayi,orta_emeliyyat_sayi
0,0,3991,16.59
1,1,1006,11.11


In [ ]:
gold_sql = sql.replace(
    "SELECT\n    churned,",
    "SELECT\n    churned,",
    1
).replace(
    "FROM customer_tx",
    "INTO gold.churn_by_transactions\nFROM customer_tx",
    1
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

In [ ]:
# 6. ƏMƏLİYYAT KANALLARINA GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
WITH kanal AS (
    SELECT
        channel,
        churned,
        COUNT(*) AS emeliyyat_sayi
    FROM silver.transactions
    GROUP BY channel, churned
)
SELECT
    channel AS kanal,
    ROUND(100.0 * SUM(CASE WHEN churned = 0 THEN emeliyyat_sayi ELSE 0 END)
        / SUM(SUM(CASE WHEN churned = 0 THEN emeliyyat_sayi ELSE 0 END)) OVER (), 2)
        AS qalanlar_faiz,
    ROUND(100.0 * SUM(CASE WHEN churned = 1 THEN emeliyyat_sayi ELSE 0 END)
        / SUM(SUM(CASE WHEN churned = 1 THEN emeliyyat_sayi ELSE 0 END)) OVER (), 2)
        AS gedenler_faiz
FROM kanal
GROUP BY channel
"""

pd.read_sql(sql, engine)

,kanal,qalanlar_faiz,gedenler_faiz
0,ATM,18.72,19.40
1,Filial,9.65,10.83
2,Mobil tətbiq,43.70,41.76
3,POS,27.92,28.01


In [ ]:
gold_sql = sql.replace(
    "FROM kanal\nGROUP BY channel",
    "INTO gold.churn_by_channel\nFROM kanal\nGROUP BY channel"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

# 7. YAŞ QRUPLARINA GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
SELECT
    CASE
        WHEN age < 30 THEN '18-29'
        WHEN age < 45 THEN '30-44'
        WHEN age < 60 THEN '45-59'
        ELSE '60+'
    END AS yas_qrupu,

    COUNT(DISTINCT customer_id) AS musteri_sayi,

    COUNT(DISTINCT CASE
        WHEN churned = 1 THEN customer_id
    END) AS geden_musteri

FROM silver.transactions
GROUP BY
    CASE
        WHEN age < 30 THEN '18-29'
        WHEN age < 45 THEN '30-44'
        WHEN age < 60 THEN '45-59'
        ELSE '60+'
    END
"""

pd.read_sql(sql, engine)

,yas_qrupu,musteri_sayi,geden_musteri
0,30-44,2318,405
1,45-59,1374,368
2,18-29,1084,149
3,60+,221,84


In [ ]:
sql = sql.replace(
    "AS geden_musteri",
    """AS geden_musteri,

    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END)
        / COUNT(DISTINCT customer_id), 2
    ) AS churn_faizi"""
)

pd.read_sql(sql, engine)

,yas_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,30-44,2318,405,17.47
1,45-59,1374,368,26.78
2,18-29,1084,149,13.75
3,60+,221,84,38.01


In [ ]:
gold_sql = sql.replace(
    "FROM silver.transactions",
    "INTO gold.churn_by_age\nFROM silver.transactions"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

# 8. BANKDA KEÇİRİLƏN MÜDDƏTƏ GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
SELECT
    tenure_years,
    COUNT(DISTINCT customer_id) AS musteri_sayi,
    COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END) AS geden_musteri,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN churned = 1 THEN customer_id END)
        / COUNT(DISTINCT customer_id), 2
    ) AS churn_faizi
FROM silver.transactions
GROUP BY tenure_years
"""

pd.read_sql(sql + " ORDER BY tenure_years", engine)

,tenure_years,musteri_sayi,geden_musteri,churn_faizi
0,0,283,40,14.13
1,1,488,120,24.59
2,2,670,145,21.64
3,3,669,153,22.87
4,4,635,128,20.16
5,5,562,108,19.22
6,6,468,95,20.30
7,7,324,63,19.44
8,8,253,45,17.79
9,9,205,37,18.05


Ən yüksək churn 1 il bankda olan müştərilərdədir — 24.59%. Ən aşağı göstərici isə 15 illik müştərilərdə — 10.45%-dir. Amma ümumilikdə churn müddət artdıqca ardıcıl azalmır.

In [ ]:
gold_sql = sql.replace(
    "FROM silver.transactions",
    "INTO gold.churn_by_tenure FROM silver.transactions"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

# 9. MÜŞTƏRİ BALANSINA GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
WITH musteriler AS (
    SELECT DISTINCT
        customer_id,
        balance,
        churned
    FROM silver.transactions
),
qruplar AS (
    SELECT
        churned,
        CASE
            WHEN balance < 2000 THEN '0-2K'
            WHEN balance < 5000 THEN '2K-5K'
            WHEN balance < 10000 THEN '5K-10K'
            WHEN balance < 20000 THEN '10K-20K'
            WHEN balance < 50000 THEN '20K-50K'
            WHEN balance < 100000 THEN '50K-100K'
            ELSE '100K+'
        END AS balans_qrupu
    FROM musteriler
)
SELECT
    balans_qrupu,
    COUNT(*) AS musteri_sayi,
    SUM(churned) AS geden_musteri,
    ROUND(
        100.0 * SUM(churned) / COUNT(*), 2
    ) AS churn_faizi
FROM qruplar
GROUP BY balans_qrupu
ORDER BY
    CASE balans_qrupu
        WHEN '0-2K' THEN 1
        WHEN '2K-5K' THEN 2
        WHEN '5K-10K' THEN 3
        WHEN '10K-20K' THEN 4
        WHEN '20K-50K' THEN 5
        WHEN '50K-100K' THEN 6
        WHEN '100K+' THEN 7
    END
"""

pd.read_sql(sql, engine)

,balans_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0-2K,1553,300,19.32
1,2K-5K,1453,285,19.61
2,5K-10K,1031,198,19.20
3,10K-20K,639,131,20.50
4,20K-50K,273,81,29.67
5,50K-100K,43,10,23.26
6,100K+,5,1,20.00


In [ ]:
gold_sql = sql.replace(
    "FROM qruplar",
    "INTO gold.churn_by_balance FROM qruplar"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

Balansı 20,000–50,000 AZN arasında olan müştərilərdə churn faizi ən yüksəkdir (29.67%). Bu qrup bank üçün həm maliyyə baxımından dəyərli, həm də müştəri itkisi baxımından diqqət tələb edən seqmentdir. Bank bu müştərilərin bankı tərk etmə səbəblərini araşdırmalı və onları saxlamaq üçün fərdi təkliflər hazırlamalıdır.

50,000 AZN-dən yuxarı balansı olan qruplarda müştəri sayı az olduğuna görə nəticələr ehtiyatla qiymətləndirilməlidir.

# 10. MAAŞ SƏVİYYƏSİNƏ GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
WITH musteriler AS (
    SELECT DISTINCT customer_id, salary, churned
    FROM silver.transactions
),
qruplar AS (
    SELECT
        churned,
        CASE
            WHEN salary < 1000 THEN '0-2K'
            WHEN salary < 3000 THEN '2K-4K'
            WHEN salary < 5000 THEN '4K-6K'
            ELSE '6K+'
        END AS maas_qrupu
    FROM musteriler
)
SELECT
    maas_qrupu,
    COUNT(*) AS musteri_sayi,
    SUM(churned) AS geden_musteri,
    ROUND(100.0 * SUM(churned) / COUNT(*), 2) AS churn_faizi
FROM qruplar
GROUP BY maas_qrupu
"""

pd.read_sql(sql, engine)

,maas_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0-2K,1807,357,19.76
1,2K-4K,2908,580,19.94
2,4K-6K,254,61,24.02
3,6K+,28,8,28.57


In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.churn_by_salary"))

gold_sql = sql.replace(
    "FROM qruplar",
    "INTO gold.churn_by_salary FROM qruplar"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

In [ ]:
pd.read_sql("SELECT * FROM gold.churn_by_salary", engine)

,maas_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0-2K,1807,357,19.76
1,2K-4K,2908,580,19.94
2,6K+,28,8,28.57
3,4K-6K,254,61,24.02


# 11. KREDİT SKORUNA GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
WITH musteriler AS (
    SELECT DISTINCT
        customer_id,
        credit_score,
        churned
    FROM silver.transactions
),
qruplar AS (
    SELECT
        churned,
        CASE
            WHEN credit_score < 600 THEN '0-599'
            WHEN credit_score < 700 THEN '600-699'
            WHEN credit_score < 800 THEN '700-799'
            ELSE '800+'
        END AS skor_qrupu
    FROM musteriler
)
SELECT
    skor_qrupu,
    COUNT(*) AS musteri_sayi,
    SUM(churned) AS geden_musteri,
    ROUND(100.0 * SUM(churned) / COUNT(*), 2) AS churn_faizi
FROM qruplar
GROUP BY skor_qrupu
"""

pd.read_sql(sql, engine)

,skor_qrupu,musteri_sayi,geden_musteri,churn_faizi
0,0-599,1206,358,29.68
1,600-699,2167,433,19.98
2,700-799,1361,179,13.15
3,800+,263,36,13.69


Aşağı kredit skorlu müştərilərin bankdan ayrılması mütləq mənfi nəticə deyil. Bu müştərilər kredit riski baxımından daha problemli ola bilərlər. Buna görə bankın məqsədi onların hamısını saxlamaq yox, risk və gəlirliliyi birlikdə qiymətləndirmək olmalıdır.

Əsas diqqət yaxşı kredit skoru olan, bank üçün gəlirli, lakin ayrılma riski daşıyan müştərilərə yönəlməlidir.

In [ ]:
gold_sql = sql.replace(
    "FROM qruplar",
    "INTO gold.churn_by_credit_score FROM qruplar"
)

with engine.begin() as conn:
    conn.execute(text(gold_sql))

: 

In [ ]:
# 12. KREDİT MÜRACİƏTİNİN RƏDD EDİLMƏSİNƏ GÖRƏ CHURN ANALİZİ

In [ ]:
sql = """
WITH musteriler AS (
    SELECT
        customer_id,
        MAX(CASE
            WHEN status = N'rədd edildi' THEN 1
            ELSE 0
        END) AS redd_alib,
        MAX(churned) AS churned
    FROM silver.loans
    WHERE customer_missing = 0
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN redd_alib = 1 THEN N'Rədd cavabı alanlar'
        ELSE N'Rədd cavabı almayanlar'
    END AS kredit_statusu,
    COUNT(*) AS musteri_sayi,
    SUM(churned) AS geden_musteri,
    ROUND(100.0 * SUM(churned) / COUNT(*), 2) AS churn_faizi
FROM musteriler
GROUP BY redd_alib
"""

pd.read_sql(sql, engine)

NameError: name 'engine' is not defined